# Incrementum - retail exploration
Exploration of the **real** Kaggle *Online Retail II* data after Python cleaning. Run the pipeline first (`clean`, `load-db`, `process --mode local`, `segments`).
Product categories here are keyword-**derived** labels; "Other / Uncertain" is left visible, not hidden.

In [ ]:
from pathlib import Path
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns, json
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CLEAN, PROC, REP = ROOT/"data/cleaned", ROOT/"data/processed", ROOT/"outputs/reports"
sns.set_theme(style="whitegrid")
def need(p):
    if not Path(p).exists(): raise FileNotFoundError(f"{p} missing - run the pipeline step that creates it")
    return p

## 1. Raw data profile and cleaning summary

In [ ]:
prof = json.load(open(need(REP/"profile_original.json")))
print("raw rows:", prof["rows"]); print(pd.Series(prof["dtypes"]))
display(pd.read_csv(need(REP/"quality_summary.csv")))
display(pd.read_csv(need(REP/"invalid_values_report.csv")))
display(pd.read_csv(need(REP/"missing_values_report.csv")))

In [ ]:
sales = pd.read_csv(need(CLEAN/"cleaned_sales.csv"), dtype={"customer_id": str},
                    usecols=["invoice","invoice_date","customer_id","stock_code","description","quantity","price","line_total","year_month","weekday","hour","department"])
print(sales.shape); sales.head()

## 2. Sales trends

In [ ]:
m = pd.read_csv(need(PROC/"monthly_sales.csv"))
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
m.plot(x="year_month", y="sales", ax=ax[0], legend=False, title="Monthly sales"); ax[0].tick_params(axis="x", rotation=60)
d = sales.assign(day=pd.to_datetime(sales["invoice_date"]).dt.date).groupby("weekday")["line_total"].sum().reindex(
    ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"])
d.plot.bar(ax=ax[1], title="Sales by weekday"); plt.tight_layout(); plt.show()

## 3. Products and categories

In [ ]:
top = pd.read_csv(need(PROC/"top_products.csv")).head(15)
sns.barplot(data=top, y="description", x="revenue"); plt.title("Top 15 products by revenue"); plt.show()
dep = pd.read_csv(need(PROC/"department_sales.csv"))
sns.barplot(data=dep, y="department", x="sales"); plt.title("Sales by derived category"); plt.show()

## 4. Customers: recency, frequency, spend

In [ ]:
f = pd.read_csv(need(PROC/"customer_features.csv"), dtype={"customer_id": str}); f = f[f["is_synthetic"] == 0]
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
for a, c in zip(ax, ["recency_days", "frequency", "monetary"]):
    sns.histplot(f[c].clip(upper=f[c].quantile(.99)), bins=40, ax=a); a.set_title(c)
plt.tight_layout(); plt.show()
print("repeat customers (2+ invoices):", f"{(f['frequency'] > 1).mean():.1%}")

## 5. Segments

In [ ]:
seg = pd.read_csv(need(PROC/"segment_summary.csv"))
display(seg[seg.segment_type.isin(["rfm_rule", "rfm_cluster"])])
seg[seg.segment_type == "category"].sort_values("customers").plot.barh(x="segment_name", y="customers", legend=False, title="Category-preference segments"); plt.show()

## 6. Products bought together (pairwise association rules)

In [ ]:
display(pd.read_csv(need(PROC/"association_rules.csv")).head(10))

## 7. Experiment (SIMULATED - control real, treatment response simulated)

In [ ]:
p = PROC/"experiment_summary.csv"
if p.exists():
    s = pd.read_csv(p).iloc[0]
    print(s["data_label"]); print(f"treatment {s.conv_treatment:.1%} vs control {s.conv_control:.1%}; diff {s.difference:+.1%} (95% CI {s.ci_low:+.1%} to {s.ci_high:+.1%}, p={s.p_value:.4f})")
else: print("run `python -m src.main experiment` first")